# Initial Data Understanding

This notebook regenerates the first-pass data quality report for the checked-in `initial_data` snapshot. It profiles file inventory, missingness, distributions, duplicates, logical review flags, and cross-dataset join coverage without modifying source CSVs.

In [ ]:
from __future__ import annotations

import csv
import subprocess
import sys
from pathlib import Path

try:
    from IPython.display import Markdown, display
except ImportError:  # Allows this notebook code to run as plain Python too.
    Markdown = lambda text: text
    display = print

ROOT = Path.cwd()
if not (ROOT / "initial_data").exists() and (ROOT.parent / "initial_data").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "initial_data"
TABLE_DIR = ROOT / "reports" / "tables"
REPORT_PATH = ROOT / "reports" / "initial_data_quality_report.md"
SCRIPT_PATH = ROOT / "scripts" / "generate_initial_data_quality_report.py"

assert DATA_DIR.exists(), f"Missing data directory: {DATA_DIR}"
assert SCRIPT_PATH.exists(), f"Missing generator script: {SCRIPT_PATH}"
ROOT

## Workflow

1. Run the standard-library generator script.
2. Verify snapshot and parsing checks in `validation_results.csv`.
3. Preview the core appendix tables.
4. Read the generated Markdown report for interpretation and next questions.

In [ ]:
result = subprocess.run(
    [sys.executable, str(SCRIPT_PATH)],
    cwd=ROOT,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
result.check_returncode()

In [ ]:
def read_rows(path: Path, limit: int | None = None) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        rows = []
        for index, row in enumerate(reader):
            if limit is not None and index >= limit:
                break
            rows.append(row)
        return rows

def markdown_table(rows: list[dict[str, str]], columns: list[str]) -> str:
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join("---" for _ in columns) + " |"]
    for row in rows:
        values = [str(row.get(column, "")).replace("|", "\\|") for column in columns]
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)

table_counts = []
for table_path in sorted(TABLE_DIR.glob("*.csv")):
    with table_path.open(newline="", encoding="utf-8") as handle:
        row_count = sum(1 for _ in handle) - 1
    table_counts.append({"table": table_path.name, "rows": str(row_count)})

display(Markdown(markdown_table(table_counts, ["table", "rows"])))

In [ ]:
validation = read_rows(TABLE_DIR / "validation_results.csv")
display(Markdown(markdown_table(validation, ["validation", "status", "detail"])))
assert all(row["status"] == "pass" for row in validation)

In [ ]:
inventory = read_rows(TABLE_DIR / "file_inventory.csv")
family_summary: dict[str, dict[str, int]] = {}
for row in inventory:
    family = row["source_family"]
    summary = family_summary.setdefault(family, {"files": 0, "logical_rows": 0})
    summary["files"] += 1
    summary["logical_rows"] += int(row["logical_rows"])

family_rows = [
    {"source_family": family, "files": str(values["files"]), "logical_rows": str(values["logical_rows"])}
    for family, values in sorted(family_summary.items())
]
display(Markdown(markdown_table(family_rows, ["source_family", "files", "logical_rows"])))

In [ ]:
cross_reference = read_rows(TABLE_DIR / "cross_reference_coverage.csv")
display(Markdown(markdown_table(
    cross_reference,
    ["relationship", "source_count", "matched_count", "unmatched_count", "coverage_pct", "sample_unmatched"],
)))

In [ ]:
duplicate_checks = [row for row in read_rows(TABLE_DIR / "duplicate_checks.csv") if row["status"] != "ok"]
quality_flags = read_rows(TABLE_DIR / "quality_flags.csv")
largest_flags = sorted(
    [row for row in quality_flags if row["status"] != "ok" and row["affected_count"].isdigit()],
    key=lambda row: int(row["affected_count"]),
    reverse=True,
)[:15]

display(Markdown("### Duplicate checks"))
display(Markdown(markdown_table(duplicate_checks, ["check_type", "item_a", "item_b", "status", "detail"])))
display(Markdown("### Largest quality flags"))
display(Markdown(markdown_table(largest_flags, ["file", "check_type", "column", "status", "affected_count", "affected_pct"])))

In [ ]:
report_lines = REPORT_PATH.read_text(encoding="utf-8").splitlines()
preview = "\n".join(report_lines[:80])
display(Markdown(preview))

## Interpretation Notes

- Treat `reports/initial_data_quality_report.md` as the primary narrative output.
- Treat `reports/tables/*.csv` as the machine-readable evidence base.
- Domain-review flags are prompts for humanitarian data interpretation, not automatic data errors.
- Do not sum HNO people values across sectors or population statuses when estimating unique people.